# Dialog servers and browser ES modules
> Start, reuse, inspect, preview, and stop FastHTML servers in Jupyter or Solveit; register browser ES modules, including versioned modules for iterative development.

In [ ]:
#| default_exp server

In [ ]:
#| export
#| hide
import inspect, json, keyword, logging, os, subprocess, re
from contextlib import asynccontextmanager
from collections.abc import Callable, AsyncIterator
from datetime import datetime
from functools import partial
from unittest.mock import patch as mock_patch
from urllib.parse import parse_qsl, urlencode, urlparse, urlunparse, quote
from starlette.middleware import Middleware
from starlette.middleware.cors import CORSMiddleware
from starlette.responses import Response
from starlette.routing import Route
from fastcore.basics import first, ifnone
from fastcore.aio import maybe_await
from fastcore.meta import delegates
from fastcore.net import is_port_free
from fastcore.xml import FT
from fasthtml.core import FastHTML
from fasthtml.jupyter import JupyUvi, JupyUviAsync, HTMX
from fasthtml.xtend import Script, Link
from dialoghelper.core import js_eval_a, read_msg
from pote.basic import Kounter
from pote.dutil import link_msg, set_var

In [ ]:
#| hide
import asyncio, socket
from multiprocessing import Queue
import fastcore.docments
from itertools import count
from starlette.testclient import TestClient
from ipykernel_helper import info_md
from fastcore.basics import setify
from fastcore.foundation import store_attr, patch
from fastcore.parallel import startproc
from fastcore.test import *
from fasthtml.common import fast_app
from pyskills import doc
from pyskills import *
from fasthtml.components import Div, Span, P
from dialoghelper.core import add_msg, iife_a, add_html_a, add_scr_a, js_run_a, del_msgs
from htmx_bridge import get_hx
from pote.logger import logger
from pote.dutil import wait_for

In [ ]:
#| exporti
kntr = Kounter()

In [ ]:
hx = get_hx()

In [ ]:
try: await maybe_await(dsrv.stop())
except BaseException: pass
dsrv=dapp=drt=None

### Choosing a helper

```python
from pote.server import ensure_server_a, serve_app, setup_mod, add_js_msg
```

- `await ensure_server_a(...)` starts or reuses a FastHTML server and returns `(srv, app, rt, p)`; by default it selects a free non-public port.
- `async with serve_app(...)` is for a scoped server that is stopped automatically, particularly in tests.
- `find_server`, `is_serving_a`, and `stop_server_a` inspect and clean up existing server state.
- `free_port(public=True)` returns first available public port.
- `p(path)` previews a route in an iframe on a public port; `p` is `None` for private ports, so they cannot be previewed.
- `setup_mod(..., versioned=True)` supports iterative browser ES-module work.
- `get_js`, `add_js_msg`, and `setup_mod` without `src` extract JavaScript from a `%%js`/`%%javascript` message.

The async APIs are normally preferable in Solveit and Jupyter. These helpers target FastHTML applications; the underlying `JupyUvi` can serve more general ASGI applications, but `pote.server` does not promise that compatibility.

## references

In [ ]:
# from inspect import getsource
# import fasthtml.jupyter
# await add_msg(getsource(fasthtml.jupyter))

`fasthtml.jupyter` provides utilities for running and displaying FastHTML/HTMX web apps **inside notebooks** (Jupyter, Solveit, etc.). Here's what the key pieces do:

**Server management**
- `nb_serve` / `nb_serve_async` — start a uvicorn server in a background thread (or async task) so it doesn't block the kernel. Waits until the server is actually ready before returning.
- `JupyUvi` / `JupyUviAsync` — convenient wrapper classes around the above, with `.start()` / `.stop()` lifecycle methods and automatic HTMX port configuration when not in Solveit.
- `wait_port_free` — blocks until a port is released (useful after `.stop()`).

**Display helpers**
- `show` — renders FastHTML components in the notebook, and triggers `htmx.process()` so htmx attributes on dynamically injected HTML are activated.
- `render_ft` — patches `FT._repr_html_` so FastHTML components auto-render with HTMX support when returned from a cell.
- `HTMX` — displays a running app in an **iframe** inside the notebook, either by URL or by rendering an `FT` component directly via a temp route.
- `htmx_config_port` — injects a JS snippet that rewrites relative htmx request paths to include the correct host/port, so htmx calls from the iframe hit the right server.

**WebSocket helper**
- `ws_client` — sets up a WebSocket-connected component displayed in an iframe, with a `send()` helper for pushing updates from Python to the browser.

In short: it bridges the gap between a notebook kernel and a live FastHTML web app, handling server lifecycle, port routing, and in-notebook rendering.

Does JupyUvi need a FastHTML app or any Starlette app will do?

---

`JupyUvi` only needs an **ASGI app**, so a plain Starlette app will do.

The constructor passes `app` straight into:

```python
uvicorn.Config(app, log_level=log_level, host=host, port=port, **kwargs)
```

so uvicorn just needs something ASGI-callable.

The only FastHTML-specific-ish part is optional `live=True`:

```python
if live: self._setup_live(app)
```

`_setup_live` assumes the app has FastHTML-style conveniences:

```python
app.hdrs.append(...)
@app.get(rt)
async def _sse(): ...
```

So:

- `JupyUvi(app)` works with any ASGI app Starlette/ FastAPI/FastHTML, etc.
- `JupyUvi(app, live=True)` expects an app with `.hdrs` and `.get`, so that's really for FastHTML-style apps unless your Starlette app provides compatible attributes.
- `HTMX(..., path=FT_or_tuple, app=app)` also assumes `app.get(route)(...)`, so that convenience path is FastHTML/Starlette-route-decorator compatible.

For a normal Starlette app:

```python
from starlette.applications import Starlette
from starlette.responses import PlainTextResponse
from starlette.routing import Route

async def home(request): return PlainTextResponse('hi')

app = Starlette(routes=[Route('/', home)])
srv = JupyUvi(app, port=8000)
```

That should work fine.

In [ ]:
doc(fastcore.docments)

# module fastcore.docments:

"""Document parameters using comments.

A *docment* documents a parameter with a trailing comment on its line in the signature, right next to the name, type, and default it describes; `docments(f)` retrieves them programmatically, so tools (docs tables, CLIs like `fastcore.script`, ...) can use them. This replaces repeating parameter names in a specially-formatted docstring, which has to be kept in sync with the signature by hand:

```python
def add(
    a:int,  # The first operand
    b=1,    # The second operand
) -> int:   # The sum
    "Add `a` to `b`"
    return a + b
test_eq(docments(add), {'a': 'The first operand', 'b': 'The second operand', 'return': 'The sum'})
```

Comment lines *above* a param also work when one line isn't enough; `typing.Annotated` string metadata is an alternative to comments; numpy-style docstring sections are merged in when present; and `docments(f, full=True)` returns each param's default, type, and docment together.

Docs: 

## helpers

In [ ]:
#| exporti
def kill_port(port):
    "Forcibly terminate local processes listening on `port`; intended only for notebook recovery"
    subprocess.run(f"lsof -ti:{port} | xargs -r kill -9", shell=True)

In [ ]:
def listen(q):
    s = socket.socket(); s.bind(('', 0)); s.listen()
    q.put(s.getsockname()[1]); s.accept()

q = Queue()
proc = startproc(lambda: listen(q), daemon=True)
port = q.get(timeout=5)
kill_port(port)
proc.join(5)

test_eq(proc.is_alive(), False)
test_is(is_port_free(port), True)

In [ ]:
#| export
def wrap_endp(
    response, # Existing request handler or fixed response value
) -> Callable:
    "Return callable handlers unchanged; otherwise return an async handler that returns `response` unchanged."
    if callable(response) and not isinstance(response, FT) and not hasattr(response, '__ft__'): return response
    async def endpoint(request): return response
    return endpoint

In [ ]:
fixed = 'ok'
ep = wrap_endp(fixed)
test_eq(await ep(None), fixed)

ft = Div('x')
test_is(await wrap_endp(ft)(None), ft)

seq = (Div('a'), Span('b'))
test_is(await wrap_endp(seq)(None), seq)

sync_handler = lambda request: 'sync'
test_is(wrap_endp(sync_handler), sync_handler)

async def async_handler(request): return 'async'
test_is(wrap_endp(async_handler), async_handler)

component = lambda: None
component.__ft__ = lambda: Div('x')
test_is(await wrap_endp(component)(None), component)

In [ ]:
json.loads(os.environ["PUBLIC_DOMAINS"]).keys()

dict_keys(['6000', '8000', '8001', '8002', '8003', '8004', '8005', '8006', '8007'])

In [ ]:
#| export
def pub_ports() -> tuple[int]:
    "Public ports reserved by this Solveit instance."
    return tuple(sorted(map(int, json.loads(os.environ.get('PUBLIC_DOMAINS', '{}')))))

In [ ]:
#| export
def free_port(
    start:int=6550, # First port to consider
    skip=(), # Ports to exclude even when available
    public:bool=False, # Restrict search to this instance's public ports?
    host:str='127.0.0.1', # Host interface used to test availability
) -> int:
    "Return the first available port; raise `RuntimeError` when none are available."
    skip, ports = set(skip), pub_ports() if public else range(start, 65536)
    if (port := first(p for p in ports if p not in skip and is_port_free(p, host))) is None: raise RuntimeError("No free port")
    return port

In [ ]:
start = free_port()
next_free = free_port(start=start+1)

test_eq(free_port(start=start), start)
test_eq(free_port(start=start, skip=[start]), next_free)
test_is(free_port(public=True) in pub_ports(), True)

with mock_patch.dict(free_port.__globals__, is_port_free=lambda *args: False):
    with expect_fail(RuntimeError, 'No free port'): free_port(public=True)

free_port(public=True)

8000

In [ ]:
#| export
def free_public_domain():
    "Public domain for the first public port that is currently free."
    return json.loads(os.environ["PUBLIC_DOMAINS"])[str(free_port(public=True))]

def pub_domain(port):
    "Public domain for `port`."
    return json.loads(os.environ['PUBLIC_DOMAINS'])[str(port)]

In [ ]:
with mock_patch.dict(free_public_domain.__globals__, free_port=lambda public: 8000):
    test_eq(free_public_domain(), json.loads(os.environ["PUBLIC_DOMAINS"])['8000'])

In [ ]:
#| export
def maybe_abs_url(url, port, sch='https', **kwargs):
    "Merge `kwargs` into `url`; for public ports except 8000, fill missing scheme and host from `sch` and `pub_domain(port)`."
    u = urlparse(url)
    qs = urlencode(dict(parse_qsl(u.query, keep_blank_values=True)) | kwargs) if kwargs else u.query
    if port == 8000 or port not in pub_ports(): u = u._replace(query=qs)
    else: u = u._replace(scheme=u.scheme or sch, netloc=u.netloc or pub_domain(port), query=qs)
    return urlunparse(u)

In [ ]:
test_eq(maybe_abs_url('', 7523), '')
test_eq(maybe_abs_url('', 8001), f"https://{pub_domain(8001)}")

test_eq(maybe_abs_url('/', 8000), '/')
test_eq(maybe_abs_url('/', 8001), f"https://{pub_domain(8001)}/")

test_eq(maybe_abs_url('/test', 7523), '/test')
test_eq(maybe_abs_url('/test', 8001), f"https://{pub_domain(8001)}/test")

test_eq(maybe_abs_url('//test.com', 8001), 'https://test.com')
test_eq(maybe_abs_url('test', 8001), f"https://{pub_domain(8001)}/test")

test_eq(maybe_abs_url('/test?a=b&c=d', 800, e='f'), f"/test?a=b&c=d&e=f")
test_eq(maybe_abs_url('/test?a=b&c=d', 8001, e='f'), f"https://{pub_domain(8001)}/test?a=b&c=d&e=f")

In [ ]:
#| export
def find_server(
    port:int|None=None, # Port to match; `None` matches any port
    all_ports:bool=False, # Return every started binding across all ports; aliases may repeat the same server
) -> tuple[JupyUvi,str]|tuple[None,None]|list[tuple[JupyUvi,str]]:
    "Find namespace-bound `JupyUvi` objects whose flags indicate they are started; no HTTP health check. First-match order follows the IPython namespace."
    import IPython
    if all_ports: port = None
    ns = IPython.get_ipython().user_ns
    res = [(v,k) for k,v in ns.items() if isinstance(v, JupyUvi) and (port is None or v.port == port) and v.server and v.server.started and not v.server.should_exit]
    return res if all_ports else res[0] if res else (None,None)

In [ ]:
srv = JupyUvi(app := FastHTML(), port=free_port())

In [ ]:
srv2,_ = find_server(srv.port)
test_is(srv2, srv)

In [ ]:
test_is(find_server(srv.port)[0], srv)
test_eq(find_server()[0], srv2)
test_eq(find_server(None, all_ports=True), find_server(all_ports=True))
test_eq(set([_[1] for _ in find_server(all_ports=True)]), {'srv', 'srv2'})

In [ ]:
srv.stop()
await asyncio.sleep(0.2)
test_eq(find_server(None, all_ports=True), [])

In [ ]:
test_eq(find_server(), (None,None))
test_eq(find_server(None, all_ports=True), [])

In [ ]:
#| export
def _conn_host(host): return '127.0.0.1' if host in ('0.0.0.0', '::') else host

def is_serving(srv:JupyUvi|None, timeout=1) -> bool:
    "Return whether `srv` appears started and `GET /` completes with a status below 500."
    if not (srv and srv.server and srv.server.started and not srv.server.should_exit and not is_port_free(srv.port, srv.host)): return False
    try:
        import httpx
        return httpx.get(f"http://{_conn_host(srv.host)}:{srv.port}/", timeout=timeout).status_code < 500
    except Exception: return False

async def is_serving_a(srv:JupyUvi|None, timeout=1) -> bool:
    "Return whether `srv` appears started and `GET /` completes with a status below 500."
    if not (srv and srv.server and srv.server.started and not srv.server.should_exit and not is_port_free(srv.port, srv.host)): return False
    try:
        import httpx
        async with httpx.AsyncClient(timeout=timeout) as cli: return (await cli.get(f"http://{_conn_host(srv.host)}:{srv.port}/")).status_code < 500
    except Exception: return False

In [ ]:
test_eq(is_serving(None), False)
test_eq(await is_serving_a(None), False)

In [ ]:
p = free_port(6551, skip=pub_ports())
srv = JupyUvi(FastHTML(), port=p)
test_eq(is_serving(srv), True)
test_eq(await is_serving_a(srv), True)
srv.stop()
await asyncio.sleep(0.2)
test_eq(is_serving(srv), False)
test_eq(await is_serving_a(srv), False)

# HTTP-500 health
p = free_port(6551, skip=pub_ports())
app = FastHTML()
@app.get('/')
def unhealthy(): return Response(status_code=500)
srv = JupyUvi(app, port=p)
test_eq((is_serving(srv), await is_serving_a(srv)), (False,False))
srv.stop()
await asyncio.sleep(0.2)

In [ ]:
#| exporti
daisyui_hdrs = (
    Link(href='https://cdn.jsdelivr.net/npm/daisyui@5', rel='stylesheet', type='text/css'),
    Script(src='https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4'),
    Link(href='https://cdn.jsdelivr.net/npm/daisyui@5/themes.css', rel='stylesheet', type='text/css'),
)
threejs_hdrs = (
    Script('{"imports":{"three":"https://cdn.jsdelivr.net/npm/three@0.165.0/build/three.module.js","three/addons/":"https://cdn.jsdelivr.net/npm/three@0.165.0/examples/jsm/"}}', type='importmap'),
)

In [ ]:
def mk_app(app_kw=None):
    if app_kw is None: app_kw = dict(hdrs=daisyui_hdrs)
    return FastHTML(**app_kw)

In [ ]:
a = mk_app({})
test_eq(all(h not in a.hdrs for h in daisyui_hdrs), True)

## server

In [ ]:
#| export
def _found_srvs(port=None):
    res = find_server(all_ports=True)
    if port is not None: res = [(srv,nm) for srv,nm in res if srv.port == port]
    srvs = {}
    for srv,nm in res:
        srvs.setdefault(id(srv), (srv, []))[1].append(nm)
    return list(srvs.values())

def stop_server(port:int|None=None) -> None:
    "Stop the discovered synchronous server on `port`, or all discovered servers when `port=None`."
    srvs = _found_srvs(port)
    if any(inspect.iscoroutinefunction(srv.stop) for srv,_ in srvs): raise RuntimeError("JupyUviAsync requires `await stop_server_a(port)`")
    for srv,nms in srvs:
        srv.stop()
        for nm in nms: set_var(nm, None)

async def stop_server_a(port:int|None=None) -> None:
    "Stop the discovered server on `port`, or all discovered servers when `port=None`."
    for srv,nms in _found_srvs(port):
        await maybe_await(srv.stop())
        for nm in nms: set_var(nm, None)

In [ ]:
p = free_port(6551, skip=pub_ports())
try: tst_srv = JupyUvi(mk_app({}), port=p)
finally: stop_server(p)
await asyncio.sleep(0.2)
test_eq(find_server(p), (None,None))
test_eq(tst_srv, None)

In [ ]:
tst_srv_a = JupyUviAsync(mk_app({}), port=p)
await tst_srv_a.start()
with expect_fail(RuntimeError, 'stop_server_a'): stop_server(p)

test_is(find_server(p)[0], tst_srv_a)
test_eq(await is_serving_a(tst_srv_a), True)

await stop_server_a(p)
test_eq(find_server(p), (None,None))
test_eq(tst_srv_a, None)

In [ ]:
pa, pb = free_port(6551, skip=pub_ports()), free_port(6551, skip=pub_ports())
try: srva, srvb = JupyUviAsync(mk_app({}), port=pa), JupyUviAsync(mk_app({}), port=pb)
finally: stop_server()
test_eq(find_server(None, all_ports=True), [])

In [ ]:
#| exporti
_CORS_slvt = Middleware(CORSMiddleware,
    allow_origin_regex=r'''https://[a-f0-9]+\.solve\.it\.com''',
    allow_methods=['*'],
    allow_headers=['*'],
)
def _app_kw(app_kw, public):
    app_kw = dict(ifnone(app_kw, {}))
    if public: app_kw['middleware'] = [*app_kw.get('middleware', []), _CORS_slvt]
    return app_kw

def get_preview(app, port=8000, host=None):
    "Return a function that displays the app in an iframe."
    if host is None: host = None if port==8000 else json.loads(os.environ['PUBLIC_DOMAINS']).get(str(port), '127.0.0.1')
    return partial(HTMX, app=app, host=host, port=port if port in (8000, None) else 443)

def srv_bundle(srv):
    if hasattr(srv.app, 'route'): return srv,srv.app,srv.app.route,get_preview(srv.app, srv.port) if srv.port in pub_ports() else None
    else: return srv,srv.app,None,None

def _server_target(srv=None, port:int|None=None, host=None, reuse=True, public=False):
    own = False
    if srv is not None: return srv,port,host,own
    if port in pub_ports(): public = True
    if public: port,host = port if port in pub_ports() else free_port(public=True, host='0.0.0.0'),'0.0.0.0'
    check_host = host or ('0.0.0.0' if port in pub_ports() else '127.0.0.1')
    if port is None: port = free_port(6551, skip=pub_ports(), host=check_host)
    found,_ = find_server(port)
    if found:
        if public and found.host not in ('0.0.0.0', '::'): raise RuntimeError(f"Server on port {port} is not publicly bound")
        if not reuse: raise RuntimeError(f"Server already running on port {port}")
        return found,port,host,own
    if not is_port_free(port, check_host): raise RuntimeError(f"Port {port} in use but no JupyUvi instance found in namespace")
    return None,port,host,True

async def _ensure_server_a(srv=None, port:int|None=None, app=None, app_kw=None, reuse=True, public=False, **jupyuvi_kw):
    srv,port,host,own = _server_target(srv, port, jupyuvi_kw.get('host'), reuse, public)
    if srv:
        if not await is_serving_a(srv): raise RuntimeError(f"Server on port {srv.port} is not serving")
        return srv,own
    if host is not None: jupyuvi_kw['host'] = host
    srv = JupyUviAsync(app if app is not None else mk_app(_app_kw(app_kw, public)), port=port, **jupyuvi_kw)
    await srv.start()
    return srv,own

def _ensure_server(srv=None, port:int|None=None, app=None, app_kw=None, reuse=True, public=False, **jupyuvi_kw):
    srv,port,host,own = _server_target(srv, port, jupyuvi_kw.get('host'), reuse, public)
    if srv:
        if not is_serving(srv): raise RuntimeError(f"Server on port {srv.port} is not serving")
        return srv,own
    if host is not None: jupyuvi_kw['host'] = host
    srv = JupyUvi(app if app is not None else mk_app(_app_kw(app_kw, public)), port=port, **jupyuvi_kw)
    return srv,own

In [ ]:
with mock_patch.dict(_server_target.__globals__,
                     pub_ports=lambda: (8000,),
                     free_port=lambda **kwargs: 8000,
                     find_server=lambda port: (None,None),
                     is_port_free=lambda *args: True):
    srv,port,host,own = _server_target(port=6551, public=True)
    test_eq((srv,port,host,own), (None,8000,'0.0.0.0',True))

with mock_patch.dict(_server_target.__globals__,
                     pub_ports=lambda: (8000,),
                     find_server=lambda port: (None,None),
                     is_port_free=lambda *args: True):
    _,port,host,own = _server_target(port=8000, host='127.0.0.1', public=True)
    test_eq((port,host,own), (8000,'0.0.0.0',True))

with mock_patch.dict(_server_target.__globals__,
                     pub_ports=lambda: (8000,),
                     free_port=lambda **kwargs: 8000,
                     find_server=lambda port: (None,None),
                     is_port_free=lambda port,host: test_eq(host, '0.0.0.0') is None):
    srv,port,host,own = _server_target(public=True)
    test_eq((port,host,own), (8000,'0.0.0.0',True))

with mock_patch.dict(_server_target.__globals__,
                     pub_ports=lambda: (8000,),
                     find_server=lambda port: (None,None),
                     is_port_free=lambda *args: True):
    _,port,host,own = _server_target(port=8000, host='127.0.0.1')
    test_eq((port,host,own), (8000,'0.0.0.0',True))

In [ ]:
srv, own = _ensure_server()
test_is(srv is not None, True)
test_is(own, True)

srv2, _ = await _ensure_server_a(port=srv.port)
test_is(srv, srv2)

In [ ]:
_srv, sym = find_server()
test_eq((_srv, sym), (srv, 'srv'))

In [ ]:
await stop_server_a()

test_is(globals()['srv'], None)
test_is(globals()['srv2'], None)
test_eq(find_server(), (None, None))
test_is(is_serving(srv2), False)

Test explicit-port start/reuse/stop.

In [ ]:
port = free_port(6551, skip=pub_ports())
srv,_ = _ensure_server(port=port)
test_eq((srv.port, is_serving(srv)), (port, True))

srv2,_ = _ensure_server(port=port)
test_is(srv2, srv)

await stop_server_a(port)


`port=None` chooses a non-public free port and starts cleanly:

In [ ]:
srv,_ = _ensure_server()
test_eq(srv.port in pub_ports(), False)
test_eq(srv.port >= 6551, True)

await stop_server_a(srv.port)

Passing the same app is accepted and reused:

In [ ]:
port = free_port(skip=pub_ports())
app = FastHTML()
srv1,_ = _ensure_server(port=port, app=app)
srv2,_ = _ensure_server(port=port, app=app)
test_is(srv1.app, srv2.app)

await stop_server_a(port)

In [ ]:
dead,_ = _ensure_server(port=None)
dead_ref = [dead]

dead.stop()
await asyncio.sleep(0.2)

test_eq(is_serving(dead_ref[0]), False)
test_fail(lambda: _ensure_server(srv=dead_ref[0]), contains='not serving', exc=RuntimeError)

dead_ref.clear()

Test the “existing server wins” contract explicitly, including that a passed `app` is ignored when a server already exists on the requested port.

In [ ]:
port = free_port(skip=pub_ports())
app1,app2 = FastHTML(),FastHTML()
srv,_ = _ensure_server(port=port, app=app1)
srv2,_ = await _ensure_server_a(port=port, app=app2)

test_is(srv2, srv)
test_is(srv2.app, app1)

await stop_server_a(port)

In [ ]:
srv,own = _ensure_server()
srv2,own2 = await _ensure_server_a(port=srv.port)

test_eq((own, own2), (True, False))
await stop_server_a(srv.port)

In [ ]:
port = free_port(skip=pub_ports())
srv,_ = await _ensure_server_a(port=port)

with expect_fail(RuntimeError, 'already running'):
    await _ensure_server_a(port=port, reuse=False)
test_eq(await is_serving_a(srv), True)

await stop_server_a(port)

In [ ]:
port = free_port(skip=pub_ports())
srv,_ = await _ensure_server_a(port=port)
ref = [srv]
await stop_server_a(port)

with expect_fail(RuntimeError, 'not serving'):
    await _ensure_server_a(srv=ref[0])

test_eq(await is_serving_a(ref[0]), False)
ref.clear()

In [ ]:
async def ensure_server_a(srv=None, port:int|None=8000, app=None, app_kw=None, host='127.0.0.1', public=False, **kwargs):
    "Return a live async server, served app, route decorator, and preview helper; existing servers win"
    srv,_ = await _ensure_server_a(srv=srv, port=port, app=app, app_kw=app_kw, reuse=True, host=host, public=public, **kwargs)
    return (*srv_bundle(srv),)

def ensure_server(srv=None, port:int|None=8000, app=None, app_kw=None, host='127.0.0.1', public=False, **kwargs):
    "Return a live server, served app, route decorator, and preview helper; existing servers win"
    srv,_ = _ensure_server(srv=srv, port=port, app=app, app_kw=app_kw, reuse=True, host=host, public=public, **kwargs)
    return (*srv_bundle(srv),)

In [ ]:
mid1 = (await read_msg(0)).id

port = free_port(skip=(6000,), public=True)
domain = json.loads(os.environ["PUBLIC_DOMAINS"])[str(port)]

srv, app, rt, p = await ensure_server_a(port=port, public=True)
test_eq((srv.port, await is_serving_a(srv)), (port, True))

@rt
def hello(): return f"Running on port {port}"

p(hello)

In [ ]:
mid2 = (await read_msg(0)).id

url = maybe_abs_url('/hello', port)# if port == 8000 else f"https://{domain}/hello"
(c := P(hx_get=url, hx_trigger='load once', id='hello01'))

In [ ]:
test_is(await wait_for(lambda: hx.eval(f'''$('#{c}').text()''')==f"Running on port {port}", timeout=5), True)

In [ ]:
await update_msg(mid1, output='')
await update_msg(mid2, output='')
await stop_server_a()

In [ ]:
try:
    port = free_port(6551, skip=pub_ports())
    srv,app,rt,p = ensure_server(port=port)
    test_is(p, None)
finally:
    await stop_server_a(port)

`ensure_server` returns the standard notebook/Solveit serving bundle: a live server, the FastHTML app it serves, the app route decorator, and a preview helper wired to that server.

Existing servers have priority. If a server is passed explicitly, it must be live and is used. Otherwise, an already-running server on the requested port is reused. If no matching server exists, a new one is started, using the provided app if any, otherwise creating a default app.

The returned preview helper always targets the returned server's port.

For the synchronous server returned here, `stop_server` stops the server and clears its notebook namespace aliases; use `await stop_server_a(...)` for asynchronous or general cleanup.

## esm loader

In [ ]:
#| export
_mod_nm_re = re.compile(r'[A-Za-z_][A-Za-z0-9_-]*\Z')
def _check_mod_nm(nm):
    if not isinstance(nm, str) or not _mod_nm_re.fullmatch(nm): raise ValueError(f"Invalid ES module name: {nm!r}")
    return nm

__mods__ = {}
def reg_mod(
    nm:str, # Kernel-global registry key
    src:str, # JavaScript module source served for this key
    v:int=0, # Numeric version; when > 0, the module is served under `<nm>?v=<v>`
) -> None:
    "Register or replace JavaScript source; entries persist in this kernel across app and server lifetimes."
    __mods__[_check_mod_nm(nm)+(f"_{v}" if v>0 else '')] = src
def _get_mod(
    nm:str, # Base module name from the request route
    v:int=0, # Requested version
    debugger:bool=False, # Prefix existing module source with `debugger;`
):
    "Return registered JavaScript as an HTTP response, or a diagnostic module when missing."
    src = __mods__.get(f'{nm}_{v}', __mods__.get(nm))
    if src is None: src,debugger = f'console.log("[{nm}] ES module version `{v}` not found");',False
    return Response(('debugger;\n' if debugger else '')+src, media_type='text/javascript')

In [ ]:
for nm in ('', '1editor', 'a b', 'a/b', 'a?b', '../editor'):
    with expect_fail(ValueError, 'Invalid ES module name'): reg_mod(nm, '')

In [ ]:
#| exporti
def _bool_q(o): return str(o).lower() in ('1','true','yes','on')

async def esm_endpoint(req):
    pp = req.path_params
    try: nm = _check_mod_nm(pp['nm'])
    except ValueError as e: return Response(str(e), status_code=400)
    res = _get_mod(nm, int(req.query_params.get('v', 0)), _bool_q(req.query_params.get('debugger', False)))
    res.headers['Access-Control-Allow-Origin'] = '*'
    return res

In [ ]:
#| exporti
esm_routes = (Route('/esm/{nm}', esm_endpoint), )

In [ ]:
def setup_loader(app=None, mods=None):
    for nm,mod in (mods or {}).items(): reg_mod(nm,mod)

def mk_app(app_kw=None):
    "Create default FastHTML app"
    if app_kw is None: app_kw = dict(hdrs=daisyui_hdrs, routes=esm_routes)
    else: app_kw = {**app_kw, 'routes': tuple(app_kw.get('routes', ()))+esm_routes}
    return FastHTML(**app_kw)

In [ ]:
__mods__.clear()
a = mk_app({})
setup_loader(mods=dict(a='console.log("a")'))
cli = TestClient(a)

test_eq(cli.get('/esm/a').text, 'console.log("a")')
test_eq(cli.get('/esm/a?v=1').text, 'console.log("a")')
test_eq(cli.get('/esm/a?v=2').text, 'console.log("a")')
reg_mod('a', 'console.log("a", 2)', 2)
test_eq(cli.get('/esm/a?v=2').text, 'console.log("a", 2)')

In [ ]:
__mods__.clear()
a = mk_app({})
cli = TestClient(a)

test_eq(cli.get('/esm/nope').headers['content-type'], 'text/javascript; charset=utf-8')
test_eq(cli.get('/esm/nope').text, 'console.log("[nope] ES module version `0` not found");')
test_eq(cli.get('/esm/nope?v=3&debugger=true').text, 'console.log("[nope] ES module version `3` not found");')

In [ ]:
#| exporti
def add_esm_routes(app):
    paths = {r.path for r in app.routes}
    for r in esm_routes:
        if r.path not in paths: app.routes.append(r)
    return app

In [ ]:
#| export
def setup_loader(
    app:FastHTML|None=None, # App receiving the ESM routes once; omit to register source without modifying an app
    mods:dict[str,str]|None=None, # Module names and sources added to the kernel-global registry
) -> None:
    "Idempotently install `/esm/{nm}` on `app` and register or replace kernel-global module sources."
    if app is not None: add_esm_routes(app)
    for nm,mod in (mods or {}).items(): reg_mod(nm,mod)

In [ ]:
#| exporti
def mk_app(app_kw=None):
    "Create default FastHTML app"
    if app_kw is None: app_kw = dict(hdrs=daisyui_hdrs)
    return add_esm_routes(FastHTML(**(app_kw or {})))

In [ ]:
__mods__.clear()
a = FastHTML()
setup_loader(a, dict(a='console.log("a")'))
cli = TestClient(a)

test_eq(cli.get('/esm/a').text, 'console.log("a")')
test_eq(sum(r.path == '/esm/{nm}' for r in a.routes), 1)
setup_loader(a)
test_eq(sum(r.path == '/esm/{nm}' for r in a.routes), 1)

In [ ]:
#| export
async def ensure_server_a(
    srv:JupyUvi|None=None, # Explicit live server; its app and port win over all creation options
    port:int|None=None, # Port used for discovery or creation; ignored when `srv` is explicit; `None` chooses a free non-public port
    app:FastHTML|None=None, # FastHTML app used only to create a server; ignored when reusing one
    mods:dict[str,str]|None=None, # Sources registered and ESM route installed after server selection, including reuse
    app_kw:dict|None=None, # Keywords for creating the default FastHTML app only; ignored when reusing one
    host:str='127.0.0.1', # loopback by default
    public=False, # set True to enforce a public port if `srv` is not given
    **kwargs, # Additional `JupyUviAsync` options used only when creating a server; ignored when reusing one
) -> tuple[JupyUvi,FastHTML,Callable|None,Callable|None]:
    "Return persistent `(srv, app, rt, p)` state, selecting explicit `srv`, matching `port`, then a new server; reuse ignores creation options but still installs `mods`."
    srv,_ = await _ensure_server_a(srv=srv, port=port, app=app, app_kw=app_kw, reuse=True, public=public, host=host, **kwargs)
    setup_loader(srv.app, mods)
    return (*srv_bundle(srv),)

def ensure_server(
    srv:JupyUvi|None=None, # Explicit live server; its app and port win over all creation options
    port:int|None=None, # Port used for discovery or creation; ignored when `srv` is explicit; `None` chooses a free non-public port
    app:FastHTML|None=None, # FastHTML app used only to create a server; ignored when reusing one
    mods:dict[str,str]|None=None, # Sources registered and ESM route installed after server selection, including reuse
    app_kw:dict|None=None, # Keywords for creating the default FastHTML app only; ignored when reusing one
    host:str='127.0.0.1', # loopback by default
    public=False, # set True to enforce a public port if `srv` is not given
    **kwargs, # Additional `JupyUvi` options used only when creating a server; ignored when reusing one
) -> tuple[JupyUvi,FastHTML,Callable|None,Callable|None]:
    "Return persistent `(srv, app, rt, p)` state, selecting explicit `srv`, matching `port`, then a new server; reuse ignores creation options but still installs `mods`."
    srv,_ = _ensure_server(srv=srv, port=port, app=app, app_kw=app_kw, reuse=True, public=public, host=host, **kwargs)
    setup_loader(srv.app, mods)
    return (*srv_bundle(srv),)

In [ ]:
__mods__.clear()
try:
    srv_a,app_a,rt_a,_ = ensure_server(mods=dict(a='console.log("a")'))
    srv_b,appB,rt_b,_ = ensure_server(mods=dict(b='console.log("b")'))
    cli = TestClient(app_a)

    test_eq(cli.get('/esm/a').text, 'console.log("a")')
    test_eq(cli.get('/esm/a?v=9&debugger=true').text, 'debugger;\nconsole.log("a")')
    test_eq(cli.get('/esm/b').text, 'console.log("b")')
finally:
    await stop_server_a(srv_a.port)

In [ ]:
await stop_server_a()

port = free_port(public=True)
srv = JupyUviAsync(FastHTML(), port=port, host='127.0.0.1')
await srv.start()

try:
    with expect_fail(RuntimeError, 'not publicly bound'):
        await ensure_server_a(port=port, public=True)
finally:
    await stop_server_a(port)

### Local dialog server

For the usual local dialog/notebook case, use default settings for non-public port selection:

```python
srv,app,rt,p = await ensure_server_a()

@rt
def hello(): return 'hello'

cli = TestClient(app)
test_eq(cli.get('/hello').text, 'hello')
```

Here `p` is `None`: private ports cannot be previewed in a Solveit iframe or used to setup ES modules loader. This is the normal mode for local notebook/dialog work and direct ASGI tests, rather than deploying a web application.

### Public server

```python
srv,app,rt,p = await ensure_server_a(public=True)

@rt
def hello(): return 'hello'
```

`await ensure_server_a(public=True)` argument is equivalent to `await ensure_server_a(port=free_port(public=True), host='0.0.0.0')`.

Then:

```python
p('/hello')
```

or:

```python
url = maybe_abs_url('/hello', srv.port)
P(hx_get=url, hx_trigger='load once')
```

Stop server:

```python
await stop_server_a(srv.port)
```

`ensure_server_a()` returns the server bundle `(srv, app, rt, p)`:

- `srv` controls the uvicorn lifecycle.
- `app` is the application actually being served.
- `rt` is its FastHTML route decorator.
- `p` previews routes through the returned server.

Selection precedence is:

1. An explicit live `srv`.
2. A live `JupyUvi` already bound to the requested port.
3. A newly created server.

When an explicit or discovered server is reused, `app`, `app_kw`, and `jupyuvi_kw` are ignored. For an explicit `srv`, its own port is authoritative and the requested `port` is ignored. `mods` are still registered, and the ESM route is installed on the reused app. `port=None` selects a free port outside Solveit’s public-port allocation.

In notebooks, prefer `await stop_server_a(srv.port)`, which supports synchronous and asynchronous servers and clears all namespace aliases to the stopped server.

### CRAFT-local server
> `ensure_server_a` is designed as reusable dialog infrastructure, rather than as a web-app deployment or a special kind of server lifetime.

A CRAFT can initialize a local server once for the dialog:

```python
srv,app,rt,_ = await ensure_server_a()
```

This creates or reuses a non-public FastHTML server and returns the `(srv, app, rt, None)` bundle. Later cells can add routes through `rt` or test `app` directly without server setup.

That covers the main ESM loader behavior:

- versioned and unversioned routes
- caller-provided app route injection
- no duplicate route injection
- `ensure_server`/`ensure_server_a` passing `mods`
- reused server receiving additional `mods`
- missing module fallback
- `debugger=true` only applying when module source exists

## test rig

In [ ]:
#| exporti
def log(msg, *args, **kwargs): print(f"[{datetime.now():%H:%M:%S}] {msg}", flush=True)

In [ ]:
log = await logger()

In [ ]:
#| exporti
class LoggerHandler(logging.Handler):
    def __init__(self, log): self.log = log; super().__init__()
    def emit(self, record):
        try: self.log(f"{(record.levelname+':'):<10}{self.format(record)}")
        except Exception: self.handleError(record)

In [ ]:
got = []
h = LoggerHandler(got.append)
h.emit(logging.LogRecord('x', logging.WARNING, '', 1, 'careful', (), None))
test_eq(got, ['WARNING:  careful'])

In [ ]:
class BadLog:
    def __call__(self, x): raise RuntimeError('boom')

h = LoggerHandler(BadLog())
h.handleError = lambda record: setattr(h, 'handled', True)
h.emit(logging.LogRecord('x', logging.INFO, '', 1, 'msg', (), None))
test_eq(h.handled, True)

In [ ]:
#| export
def setup_uvi_logging(
    _log:Callable[[str],object]|None=None, # Callable receiving formatted log lines; defaults to this module's `log`
    level:int=logging.INFO, # Level assigned to the uvicorn, uvicorn.error, and uvicorn.access loggers
) -> Callable[[],None]:
    "Route uvicorn logs to `_log`; return caller-owned cleanup and use `log_config=None` to preserve the setup."
    _log = log if _log is None else _log
    _lh = LoggerHandler(_log)
    names = ('uvicorn', 'uvicorn.error', 'uvicorn.access')
    prev = {}
    for name in names:
        lg = logging.getLogger(name)
        prev[name] = lg.handlers[:],lg.propagate,lg.level,lg.disabled
        lg.handlers,lg.propagate,lg.disabled = [_lh],False,False
        lg.setLevel(level)
    def cleanup():
        for name,(handlers,propagate,level,disabled) in prev.items():
            lg = logging.getLogger(name)
            lg.handlers,lg.propagate,lg.disabled = handlers,propagate,disabled
            lg.setLevel(level)
    return cleanup

Verify all three uvicorn loggers are changed and then restored.

In [ ]:
names = ('uvicorn','uvicorn.error','uvicorn.access')
logs = [logging.getLogger(o) for o in names]
old = [(o.handlers[:],o.propagate,o.level,o.disabled) for o in logs]
cleanup = setup_uvi_logging(lambda x: None, level=logging.WARNING)

test_eq([o.level for o in logs], [logging.WARNING]*3)
test_eq([o.propagate for o in logs], [False]*3)
test_eq([o.disabled for o in logs], [False]*3)
test_eq([len(o.handlers) for o in logs], [1]*3)

cleanup()
test_eq([(o.handlers,o.propagate,o.level,o.disabled) for o in logs], old)

In [ ]:
#| export
@asynccontextmanager
async def serve_app(
    app:FastHTML|None=None, # App to serve; omit to create a default app
    port:int|None=None, # Exact port to bind; omit to select a free private/public port
    after_start:Callable[[JupyUvi],object]|None=None, # Sync or async callback after server startup
    before_stop:Callable[[JupyUvi],object]|None=None, # Sync or async callback before server shutdown
    mods:dict[str,str]|None=None, # JavaScript module sources registered for the app
    app_kw:dict|None=None, # Keywords used only when creating the default app
    public:bool=False, # Select a public port and public server binding
    **kwargs, # Additional options passed when creating `JupyUviAsync`
) -> AsyncIterator[JupyUvi]:
    "Yield a newly created server in a temporary notebook namespace scope, then stop it."
    if port is None:
        host = '0.0.0.0' if public else '127.0.0.1'
        port = free_port(public=True, host=host) if public else free_port(6551, skip=pub_ports(), host=host)
    public = public or port in pub_ports()
    host = '0.0.0.0' if public else '127.0.0.1'
    with mock_patch.dict(get_ipython().user_ns, {}):
        srv,_ = await _ensure_server_a(app=app, port=port, app_kw=app_kw, public=public, reuse=False, **kwargs)
        setup_loader(srv.app, mods)
        try:
            if after_start: await maybe_await(after_start(srv))
            yield srv
        finally:
            try:
                if before_stop: await maybe_await(before_stop(srv))
            finally: await maybe_await(srv.stop())

In [ ]:
ports = []
globals().pop('test_srv', None)

async with serve_app() as test_srv:
    ports.append(test_srv.port)
    test_eq(await is_serving_a(test_srv), True)

test_eq(is_port_free(ports[0]), True)
test_eq('test_srv' in globals(), False)

The first test establishes the fresh-server and temporary-alias contracts. Next pin down the central safety property: an existing server remains unchanged.

- `ensure_server_a` is **persistent and reusable**: an existing server on the selected port wins.
- `serve_app` is **scoped and owned**: it creates a fresh server, yields it, and stops exactly that server on exit.

In [ ]:
try:
    pre_srv,_,_,_ = await ensure_server_a()

    async with serve_app() as test_srv:
        test_is(test_srv is pre_srv, False)
        test_eq(await is_serving_a(pre_srv), True)

    test_eq(await is_serving_a(pre_srv), True)
    test_eq('test_srv' in globals(), False)
finally:
    await stop_server_a(pre_srv.port)

In [ ]:
try:
    pre_srv = JupyUvi(fast_app(), host='127.0.0.1', port=free_port(skip=pub_ports()))
    test_srv = []

    async with serve_app() as pre_srv:
        test_srv.append(pre_srv)
        test_eq(await is_serving_a(test_srv[0]), True)

    test_is(isinstance(test_srv[0], JupyUviAsync), True)
    test_is(pre_srv is test_srv[0], False)
    test_eq(await is_serving_a(test_srv[0]), False)
finally:
    await stop_server_a(pre_srv.port)

The canonical-server test confirms `serve_app` creates a distinct server and leaves the existing dialog server live. Next test the no-reuse rule for an occupied explicit port:

In [ ]:
port = free_port(6551, skip=pub_ports())
errs = []

async with serve_app(port=port):
    try:
        async with serve_app(port=port): pass
    except RuntimeError as e: errs.append(str(e))

test_eq(errs, [f"Port {port} in use but no JupyUvi instance found in namespace"])
test_eq(is_port_free(port), True)

The supplied-app contract is covered. Next verify callback order and cleanup:

In [ ]:
events,ports = [],[]

async def after(srv): events.append(('after', srv.port))
def before(srv): events.append(('before', srv.port))

async with serve_app(after_start=after, before_stop=before) as test_srv:
    ports.append(test_srv.port)
    test_eq(events, [('after', test_srv.port)])

test_eq(events, [('after', ports[0]), ('before', ports[0])])
test_eq(is_port_free(ports[0]), True)

The normal callback sequence is covered. Next verify that an `after_start` failure still runs `before_stop` and releases the port:

In [ ]:
events,ports,err = [],[],None

async def after(srv):
    ports.append(srv.port)
    events.append('after')
    raise RuntimeError('after boom')

def before(srv): events.append('before')

try:
    async with serve_app(after_start=after, before_stop=before): pass
except RuntimeError as e: err = e

test_eq(str(err), 'after boom')
test_eq(events, ['after', 'before'])
test_eq(is_port_free(ports[0]), True)

The callback-failure path is covered. Now verify a failure in the context body still stops the server and removes the temporary alias:

In [ ]:
ports,err = [],None
globals().pop('test_srv', None)

try:
    async with serve_app() as test_srv:
        ports.append(test_srv.port)
        raise ValueError('body boom')
except ValueError as e: err = e

test_eq(str(err), 'body boom')
test_eq(is_port_free(ports[0]), True)
test_eq('test_srv' in globals(), False)

The body-failure path is covered. Next verify `before_stop` failures do not prevent shutdown:

In [ ]:
ports,err = [],None

def before(srv):
    ports.append(srv.port)
    raise RuntimeError('before boom')

try:
    async with serve_app(before_stop=before): pass
except RuntimeError as e: err = e

test_eq(str(err), 'before boom')
test_eq(is_port_free(ports[0]), True)

This distinguishes app/server isolation from module-registry isolation.

In [ ]:
setup_uvi_logging()
await log.show(True)

[15:45:44.146] INFO:     Started server process [2607]
[15:45:44.184] INFO:     Waiting for application startup.
[15:45:44.217] TRACE:    ASGI [1] Started scope={'type': 'lifespan', 'asgi': {'version': '3.0', 'spec_version': '2.0'}, 'state': {}}
[15:45:44.250] TRACE:    ASGI [1] Receive {'type': 'lifespan.startup'}
[15:45:44.285] TRACE:    ASGI [1] Send {'type': 'lifespan.startup.complete'}
[15:45:44.317] INFO:     Application startup complete.
[15:45:44.349] INFO:     Uvicorn running on http://0.0.0.0:6551 (Press CTRL+C to quit)
[15:45:44.480] INFO:     Shutting down
[15:45:44.618] INFO:     Waiting for application shutdown.
[15:45:44.650] TRACE:    ASGI [1] Receive {'type': 'lifespan.shutdown'}
[15:45:44.683] TRACE:    ASGI [1] Send {'type': 'lifespan.shutdown.complete'}
[15:45:44.718] TRACE:    ASGI [1] Completed
[15:45:44.751] INFO:     Application shutdown complete.
[15:45:44.783] INFO:     Finished server process [2607]


In [ ]:
async with serve_app(log_level='trace', log_config=None) as test_srv:
    app = test_srv.app

### Temporary server and testing

Use `serve_app` when a server should exist only for a block:

```python
async with serve_app() as test_srv:
    @test_srv.app.get('/hello')
    def hello(): return 'hello'

    cli = TestClient(test_srv.app)
    test_eq(cli.get('/hello').text, 'hello')
```

`serve_app` always starts a fresh server and owns its lifetime. It stops that server after normal exit, setup or body errors, and cancellation. An explicitly requested port must be free; an occupied port raises rather than reusing its existing server.

The context temporarily scopes the notebook namespace. A new alias such as `test_srv` is removed on exit; an existing name temporarily rebound by the context is restored.

Route changes made to `test_srv.app` persist on a caller-supplied app.

The context yields the live server; use `test_srv.app`, `test_srv.port`, and its other attributes inside the block. `port=None` selects a free non-public port by default.

### Uvicorn logging

`setup_uvi_logging` routes the `uvicorn`, `uvicorn.error`, and `uvicorn.access` loggers to a callable and returns a cleanup function that restores their previous configuration.

Pass `log_config=None` when starting uvicorn; otherwise uvicorn may replace the custom logger configuration.

```python
cleanup = setup_uvi_logging(my_log)
try:
    async with serve_app(log_config=None) as srv:
        ...
finally:
    cleanup()
```

Logging configuration is process-global rather than server-owned, so the caller is responsible for cleanup.

## inject mod
Turn JavaScript stored in dialog messages into browser-loadable ES modules.

In [ ]:
#| export
async def get_js(
    id:str|None=None, # JavaScript message id; defaults to the preceding message
    dname:str='', # Optional dialog name containing the message
) -> str|None:
    "Strip the leading `%%js` or `%%javascript` line from a dialog message and return its remaining source."
    fl,_,src = (await read_msg(n=0 if id else -1, id=id, dname=dname)).content.partition('\n')
    if fl.lower() not in ('%%javascript', '%%js'): return None
    return src

In [ ]:
test_eq(await get_js(), 'a')

In [ ]:
test_is(await get_js(), None)

In [ ]:
#| exporti
def _py_assign(nm:str, src:str):
    "Return a Python assignment that round-trips `src`; reject invalid variable names."
    if not nm.isidentifier() or keyword.iskeyword(nm): raise ValueError(f"Invalid Python identifier: {nm!r}")
    return f'{nm} = {src!r}'

In [ ]:
src = r"""const single = \"'\"
const triple = "'''"
const path = String.raw`C:\tmp\file`
const tmpl = `line 1
line 2`
""" + "\\"
ns = {}
exec(_py_assign('_src', src), ns)
test_eq(ns['_src'], src)

for nm in ('not-valid', 'class', ''):
    with expect_fail(ValueError, 'Invalid Python identifier'): _py_assign(nm, src)

In [ ]:
#| export
@delegates(link_msg, but='content,msg_type')
async def add_js_msg(
    nm:str, # Valid Python identifier assigned to safely serialized source; invalid names raise `ValueError`
    id:str=None, # JavaScript source message; defaults to the preceding message and does not change the linkage anchor
    dname:str='', # Dialog to get info for; the linked code message is created in the current dialog
    **kwargs, # Delegated `link_msg` placement, execution, and display options; output is always a code message
) -> str:
    "Extract JavaScript from a dialog message and create or update a linked Python code message containing a safely serialized assignment."
    if (src := await get_js(id, dname)) is not None:
        return await link_msg(_py_assign(nm, src), msg_type='code', **kwargs)
    raise ValueError(f"Message #{id or (await read_msg()).id} does not contain Javascript source")

In [ ]:
await add_js_msg('_test_js', i_collapsed=True);

In [ ]:
_test_js = "const test = {'hello':'world'}\nglobalThis._test = test\nexport {test}"
#| linkedto: d954aa42

In [ ]:
v,dname = None,find_dname()
q = (*filter(None, (f"v={v}" if v is not None else '', f"dlg_name={quote(dname.lstrip('/'), safe='')}" if dname else '')),)
'&'.join(q)

'dlg_name=prj%2Fpote%2Fnbs%2F02_server'

In [ ]:
v=1
q = (*filter(None, (f"v={v}" if v is not None else '', f"dlg_name={quote(dname.lstrip('/'), safe='')}" if dname else '')),)
'&'.join(q)

'v=1&dlg_name=prj%2Fpote%2Fnbs%2F02_server'

In [ ]:
nm = 'a'
q = (*filter(None, (f"v={v}" if v is not None else '', f"dlg_name={quote(dname.lstrip('/'), safe='')}" if dname else '')),)
url = f'/esm/{nm}' + ('?' if q else '') + '&'.join(q)
url

'/esm/a?v=1&dlg_name=prj%2Fpote%2Fnbs%2F02_server'

In [ ]:
nm = 'a'
q = []
url = f'/esm/{nm}' + ('?' if q else '') + '&'.join(q)
url

'/esm/a'

In [ ]:
@delegates(serve_app)
async def load_mod(
    nm:str, # Route-safe module name previously registered with `reg_mod`
    v:int=0, # Numeric version; when > 0, will load `/esm/<nm>?v=<v>`
    dname:str='', # Dialog owning the module; will be appended as query param if present
    **kwargs
) -> object:
    "Ensure the server, import `/esm/<nm>` in the active Solveit browser, then release any temporary server; browser and registry state persist."
    q = (*filter(None, (f"v={v}" if v > 0 else '', f"dlg_name={quote(dname.lstrip('/'), safe='')}" if dname else '')),)
    url = f'/esm/{nm}' + ('?' if q else '') + '&'.join(q)
    async with serve_app(**kwargs) as _: return await js_eval_a(f"return await import({json.dumps(url)})")

In [ ]:
reg_mod('demo', 'document.documentElement.dataset.demo = "loaded"; export const status = "loaded"')

res = await load_mod('demo', public=True)
test_eq(hx.eval('document.documentElement.dataset.demo'), 'loaded')
res

```python
{ 'data_id': '87e04ae8-4cdb-4d86-833d-d1c9a1bab14a',
  'result': {'status': 'loaded'}}
```

In [ ]:
reg_mod('demo', 'document.documentElement.dataset.demo = "version-2"; export const status = "version-2"', v=2)

await load_mod('demo', v=2, public=True)
test_eq(hx.eval('document.documentElement.dataset.demo'), 'version-2')

In [ ]:
test_eq((await js_eval_a('''const mod = await import('/esm/demo?v=2')\nreturn mod''')).result, {'status': 'version-2'})

Since the module is cached, we no longer need a running server.

In [ ]:
async def load_mod(
    nm:str, # Route-safe module name previously registered with `reg_mod`
    v:int=0, # # Numeric version; when > 0, will load `/esm/<nm>?v=<v>`
    dname:str='', # Dialog owning the module; will be appended as query param if present
) -> object:
    "Import `/esm/<nm>` in the active Solveit browser; return URL and module result. Need at least one public port free."
    _check_mod_nm(nm)
    async with serve_app(public=True) as srv:
        # q = (*filter(None, (f"v={v}" if v > 0 else '', f"dlg_name={quote(dname.lstrip('/'), safe='')}" if dname else '')),)
        # base = '' if srv.port == 8000 else f"https://{pub_domain(srv.port)}"
        # url = f'{base}/esm/{nm}' + ('?' if q else '') + '&'.join(q)
        # q = ({'v':v} if v > 0 else {}) | ({'dname':quote(dname.lstrip('/'), safe='')} if dname else {})
        q = dict(filter(None, (('v',v) if v > 0 else None, ('dlg_name',quote(dname.lstrip('/'), safe='')) if dname else None)))
        url = maybe_abs_url(f"/esm/{nm}", srv.port, **q)
        return url, await js_eval_a(f"return await import({json.dumps(url)})")

Since we need a public server to load the module the first time, `load_mod` ensures that a public one is available. It returns now the URL; for ports other than 8000, we need a full URL.

In [ ]:
reg_mod('test_load_mod', 'window.__test_load_mod = 7; export const answer = 7')

url,mod = await load_mod('test_load_mod')
print(url)

test_eq(mod['result']['answer'], 7)
test_eq((await js_eval_a('return window.__test_load_mod'))['result'], 7)

/esm/test_load_mod


In [ ]:
reg_mod('test_load_mod_error', 'throw new Error("test module failure")')

with expect_fail(DialogAPIError, 'Timed out'): await load_mod('test_load_mod_error')

In [ ]:
#| export
async def load_mod(
    nm:str, # Route-safe module name previously registered with `reg_mod`
    v:int=0, # Numeric version; when > 0, will load `/esm/<nm>?v=<v>`
    dname:str='', # Dialog owning the module; will be appended as query param if present
) -> object:
    "Import a registered ES module in the active Solveit browser and return the url and its exports (as JSON)."
    _check_mod_nm(nm)
    async with serve_app(public=True) as srv:
        q = dict(filter(None, (('v',v) if v > 0 else None, ('dlg_name',quote(dname.lstrip('/'), safe='')) if dname else None)))
        url = maybe_abs_url(f"/esm/{nm}", srv.port, **q)
        res = await js_eval_a(f'''try {{ return await import({json.dumps(url)}) }} catch (e) {{ return {{__load_mod_error__: `${{e.name}}: ${{e.message}}`}} }}''')
    if err := res['result'].get('__load_mod_error__'): raise RuntimeError(err)
    return url, res

`load_mod` fails fast now. It catches an import rejection in the browser, returns its name/message through the existing result channel, then raises immediately in Python.

The successful return value is unchanged. A module that throws `new Error("test module failure")` should now promptly raise `RuntimeError: Error: test module failure`.

In [ ]:
reg_mod('test_load_mod_error', 'throw new Error("test module failure")', v=1)

with expect_fail(RuntimeError, 'Error: test module failure'):
    await load_mod('test_load_mod_error', 1)

In [ ]:
reg_mod('test_load_mod', 'window.__test_load_mod = 13; export const answer = 13', v=1)

url,mod = await load_mod('test_load_mod', 1)
print(url)

test_eq(mod['result']['answer'], 13)
test_eq((await js_eval_a('return window.__test_load_mod'))['result'],13)

/esm/test_load_mod?v=1


In [ ]:
#| export
async def setup_mod(
    nm:str, # Stable base ESM name
    src:str|None=None, # JavaScript source; defaults to the preceding JS message
    versioned:bool=False, # Use a kernel-local counter for `?v=<version>` to bypass browser cache
    dname:str='', # Dialog owning the module; will be appended as query param if present
) -> tuple[str,object]:
    "Register stable or versioned JavaScript, import it, and return `(url, browser_result)`."
    _check_mod_nm(nm)
    if src is None: src = await get_js()
    if src is None: raise ValueError(f"Message #{(await read_msg()).id} does not contain Javascript source")
    v = kntr(nm) if versioned else 0
    reg_mod(nm, src, v)
    return await load_mod(nm, v, dname=dname)

In [ ]:
url,mod = await setup_mod('test_setup_mod', 'window.__setup_mod = 17; export const answer = 17')

test_eq(mod['result']['answer'], 17)
test_eq((await js_eval_a('return window.__setup_mod'))['result'], 17)
url

'/esm/test_setup_mod'

In [ ]:
url1,res1 = await setup_mod('test_setup_mod', 'export const answer = 1', versioned=True)
url2,res2 = await setup_mod('test_setup_mod', 'export const answer = 2', versioned=True)

test_eq((res1['result']['answer'], res2['result']['answer']), (1,2))
test_eq(url1, '/esm/test_setup_mod?v=1')
test_eq(url2, '/esm/test_setup_mod?v=2')

In [ ]:
url,res = await setup_mod('test_setup_mod_msg')

test_eq(res['result']['answer'], 23)

### Browser ES modules

Register and import JavaScript in the active Solveit browser with:

```python
src = 'export const answer = 42'
url,res = await setup_mod('editor', src, versioned=True)
```

With `versioned=True`, repeated calls register successive names such as `editor_1` and `editor_2`, and import distinct URLs such as `/esm/editor?v=1`. Use this during iterative development to bypass the browser’s ES-module cache.

The module registry belongs to the current Python kernel, not to an app or server. Registering an existing key replaces its source, and registered entries survive server shutdown. There is currently no public operation for unregistering an entry or clearing the registry.

Registered modules are executable browser code and may be served through the app’s public HTTP endpoint. Register only trusted source; module names are not an authorization boundary.

An unversioned name provides a stable URL. Replacing its registered source changes future HTTP responses, but a browser that has already imported that URL may continue using its cached module.

If given, `dname` is URL-encoded and forwarded as the `dlg_name` query parameter during the browser import. It does not affect module-registry lookup; browser-side code can use it only if it reads that parameter.

`setup_mod` and `load_mod` create a temporary public server for the browser import, then stop it once the import completes. Port 8000 produces a relative import URL; another selected public port produces a full URL using its assigned public domain. The module registry and browser module cache persist after that temporary server stops.

JavaScript can also come from a dialog message whose first line is `%%js` or `%%javascript`:

- `get_js` returns the source without the magic line.
- `setup_mod` without `src` extracts and imports the preceding JavaScript message.
- `add_js_msg` creates linked Python state containing the extracted source.

A missing registry entry is served as a valid diagnostic JavaScript module rather than an HTTP error, so an import may succeed without providing the expected exports.

###### start-section

Let's test programatic import in the browser: `const {...} = await import('/esm/x?v=...')`

In [ ]:
# await setup_mod('x', 'export const x=42,y="ok"', True)

In [ ]:
# res = await js_eval_a("""const {x,y} = await import('/esm/x?v=1'); return {x,y}""")
# test_eq(res['result'], dict(x=42, y='ok'))

How could we use `import('x')` instead of `import('/esm/x?v=1')`

---

A bare specifier such as `import('x')` needs an **import map** in the document where the JavaScript runs. The existing `threejs_hdrs` is the same pattern.

For a fixed module version:

```python
app.hdrs.append(Script(json.dumps(dict(imports=dict(x='/esm/x?v=1'))), type='importmap'))
```

Then, after the browser has loaded that document:

```python
res = await js_eval_a("""const {x,y} = await import('x'); return {x,y}""")
test_eq(res['result'], dict(x=42, y='ok'))
```

For the unversioned, current module, map it to `/esm/x` instead:

```python
app.hdrs.append(Script(json.dumps(dict(imports=dict(x='/esm/x'))), type='importmap'))
```

Native import maps are fixed once module loading has begun. Thus `import('x')` is useful for a stable module identity, but versioned hot-reloads should keep using distinct URLs such as `/esm/x?v=1`, `/esm/x?v=2` (or use a shim that supports mutable import maps).

###### end-section

# pyskill

In [ ]:
import pote.server

list_pyskills()['pote.server']

'Start, reuse, inspect, preview, and stop FastHTML servers in Jupyter or Solveit; register browser ES modules, including versioned modules for iterative development.'

In [ ]:
doc(pote.server)

# module pote.server:

"""Start, reuse, inspect, preview, and stop FastHTML servers in Jupyter or Solveit; register browser ES modules, including versioned modules for iterative development.

### Choosing a helper

```python
from pote.server import ensure_server_a, serve_app, setup_mod, add_js_msg
```

- `await ensure_server_a(...)` starts or reuses a FastHTML server and returns `(srv, app, rt, p)`; by default it selects a free non-public port.
- `async with serve_app(...)` is for a scoped server that is stopped automatically, particularly in tests.
- `find_server`, `is_serving_a`, and `stop_server_a` inspect and clean up existing server state.
- `free_port(public=True)` returns first available public port.
- `p(path)` previews a route in an iframe on a public port; `p` is `None` for private ports, so they cannot be previewed.
- `setup_mod(..., versioned=True)` supports iterative browser ES-module work.
- `get_js`, `add_js_msg`, and `setup_mod` without `src` extract JavaScript from a `

In [ ]:
doc(pote.server.ensure_server)

def ensure_server(
    srv:fasthtml.jupyter.JupyUvi | None=None, # Explicit live server; its app and port win over all creation options
    port:int | None=None, # Port used for discovery or creation; ignored when `srv` is explicit; `None` chooses a free non-public port
    app:fasthtml.core.FastHTML | None=None, # FastHTML app used only to create a server; ignored when reusing one
    mods:dict[str, str] | None=None, # Sources registered and ESM route installed after server selection, including reuse
    app_kw:dict | None=None, # Keywords for creating the default FastHTML app only; ignored when reusing one
    host:str='127.0.0.1', # loopback by default
    public:bool=False, # set True to enforce a public port if `srv` is not given
    **jupyuvi_kw
)->tuple: # Additional `JupyUvi` options used only when creating a server; ignored when reusing one"""Return persistent `(srv, app, rt, p)` state, selecting explicit `srv`, matching `port`, then a new server; reuse ignores creation option

In [ ]:
doc(pote.server.serve_app)

def serve_app(
    app:fasthtml.core.FastHTML | None=None, # App to serve; omit to create a default app
    port:int | None=None, # Exact port to bind; omit to select a free private/public port
    after_start:collections.abc.Callable[[fasthtml.jupyter.JupyUvi], object] | None=None, # Sync or async callback after server startup
    before_stop:collections.abc.Callable[[fasthtml.jupyter.JupyUvi], object] | None=None, # Sync or async callback before server shutdown
    mods:dict[str, str] | None=None, # JavaScript module sources registered for the app
    app_kw:dict | None=None, # Keywords used only when creating the default app
    public:bool=False, # Select a public port and public server binding
    **jupyuvi_kw
)->AsyncIterator: # Additional options passed when creating `JupyUviAsync`"""Yield a newly created server in a temporary notebook namespace scope, then stop it."""

In [ ]:
doc(pote.server.setup_mod)

async def setup_mod(
    nm:str, # Stable base ESM name
    src:str | None=None, # JavaScript source; defaults to the preceding JS message
    versioned:bool=False, # Use a kernel-local counter for `?v=<version>` to bypass browser cache
    dname:str='', # Dialog owning the module; will be appended as query param if present
)->tuple:"""Register stable or versioned JavaScript, import it, and return `(url, browser_result)`."""

# export -

In [ ]:
#|hide
#|eval: false
from pote.flakes import show_flakes
await show_flakes()

No warnings to report

In [ ]:
#|hide
#|eval: false
from pote.dialog import dlg_export
await dlg_export()

True